<a href="https://colab.research.google.com/github/haydenarnold/CMV2/blob/main/%5Btemplate%5D_MoralBERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd

from huggingface_hub import PyTorchModelHubMixin
from transformers import AutoModel, AutoTokenizer
from tqdm.auto import tqdm

In [8]:
from google.colab import userdata
import os

# Colab Secrets에서 HF_TOKEN을 불러옵니다.
# Colab Secrets에 'HF_TOKEN'이라는 이름으로 Hugging Face 토큰을 저장했는지 확인하세요.
HF_TOKEN = userdata.get('HF_TOKEN')

In [9]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("사용 장치:", device)

사용 장치: cuda


In [10]:
MODEL_BASE = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_BASE)

class MoralBERTModel(
    nn.Module,
    PyTorchModelHubMixin,
    pipeline_tag="text-classification",
    license="mit",
):
    def __init__(self, bert_model, moral_label=2):
        super().__init__()

        self.bert = bert_model

        self.invariant_trans = nn.Linear(768, 768)

        self.moral_classification = nn.Sequential(
            nn.Linear(768, 768),
            nn.ReLU(),
            nn.Linear(768, moral_label)
        )

    def forward(
        self,
        input_ids,
        attention_mask,
        token_type_ids=None
    ):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )

        # [CLS] token representation
        pooled_output = outputs.last_hidden_state[:, 0, :]

        pooled_output = self.invariant_trans(pooled_output)
        logits = self.moral_classification(pooled_output)

        return logits

In [11]:
# 기본 moralBERT 간단 예시
# foundation = "authority"

# repo_name = (
#     f"vjosap/moralBERT-predict-{foundation}-in-text"
# )

# bert_model = AutoModel.from_pretrained(MODEL_BASE)

# model = MoralBERTModel.from_pretrained(
#     repo_name,
#     bert_model=bert_model
# )

# model.to(device)
# model.eval()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


MoralBERTModel(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_af

### Dataset Load

In [6]:
df = pd.read_feather('/content/drive/MyDrive/Colab Notebooks/reddit_CMV/MFT(cmv ver.2)/dataset/dataset_by_year/dataset_2019.feather')
df.head()

,thread_id,OP,post_id,thread_size,comment_depth,is_delta,thread_pattern,ChallengerOP,comment_id,author,...,post_score,title,Topic,Title,Politicality,length_post,delta_ratio,processed_body,processed_post,post_year
0,67aa4f330bc679138ff86165,top_kek_nice_meme,abdk3z,3.0,1,0,COC,C,6785425bf5757a629b9c5258,ItsPandatory,...,11,CMV: Bernie Sanders is an actual socialist,85,Bernie Sanders,1,100,0.0,I think you might have a strong argument from ...,A frequent talking point of leftist discussion...,2019
1,67aa4f330bc679138ff86165,top_kek_nice_meme,abdk3z,3.0,3,0,COC,C,6785425bf5757a629b9c5283,ItsPandatory,...,11,CMV: Bernie Sanders is an actual socialist,85,Bernie Sanders,1,100,0.0,I need some more information to make a specifi...,A frequent talking point of leftist discussion...,2019
2,67aa4f330bc679138ff8616b,top_kek_nice_meme,abdk3z,1.0,1,0,C,C,6785425bf5757a629b9c5300,MyHope4usAll,...,11,CMV: Bernie Sanders is an actual socialist,85,Bernie Sanders,1,100,0.0,"He states in the same article, ""What we were s...",A frequent talking point of leftist discussion...,2019
3,67aa4f330bc679138ff8616c,top_kek_nice_meme,abdk3z,1.0,1,0,C,C,6785425bf5757a629b9c5684,pillbinge,...,11,CMV: Bernie Sanders is an actual socialist,85,Bernie Sanders,1,100,0.0,"When you say ""an actual socialist"", there is a...",A frequent talking point of leftist discussion...,2019
4,67aa4f330bc679138ff8616d,top_kek_nice_meme,abdk3z,1.0,1,0,C,C,6785429ef5757a629ba19737,CDWEBI,...,11,CMV: Bernie Sanders is an actual socialist,85,Bernie Sanders,1,100,0.0,> Social (public) ownership of MoP\n\nWhat he...,A frequent talking point of leftist discussion...,2019


## 문장 단위(sentence-aware) chunking

지금까지의 `predict_single_text_chunked`는 토큰 개수만 보고 148개씩 기계적으로 잘랐기 때문에, 문장/절 중간이 잘릴 수 있다 (예: "stealing is wrong because..."에서 "because" 뒤가 다음 청크로 넘어감). 도덕적 수사는 보통 문장 단위로 완결되므로, **문장 경계를 지키면서** 각 청크가 150 토큰 이내가 되도록 문장을 그리디하게 묶는다.

- `nltk.sent_tokenize`로 문장을 나눈 뒤, 토큰 예산(148 = 150 - `[CLS]`/`[SEP]`)을 넘지 않는 선에서 문장을 순서대로 채워 넣는다.
- 청크 경계에서 문맥이 완전히 끊기지 않도록 `sentence_overlap`개 문장을 다음 청크 시작에 다시 포함시킨다 (토큰 단위 `stride`의 문장 버전).
- **엣지 케이스**: 문장 하나가 148토큰을 넘는 run-on sentence(쉼표만 있고 마침표가 드문 Reddit 댓글에서 실제로 나옴)는 그 문장 하나만 `_split_long_sentence`로 토큰 단위 sliding-window fallback을 적용한다. (처음엔 이 문장을 통째로 넣고 `truncation=True`에 맡기려 했는데, 그러면 뒷부분이 조용히 잘려나가는 걸 검증 중 발견해서 수정함 — 301토큰짜리 단일 문장 테스트에서 뒤 153토큰이 유실되는 걸 확인했다.)
- 집계는 이전과 동일하게 **max-pooling**을 사용한다.

In [12]:
import nltk
nltk.download("punkt_tab", quiet=True)
from nltk.tokenize import sent_tokenize


def _split_long_sentence(
    sentence: str,
    tokenizer,
    max_length: int,
    stride: int,
) -> list[str]:
    """문장 하나가 max_length보다 긴 극단적 케이스(run-on sentence 등)를 위한 토큰 단위 fallback"""
    content_capacity = max_length - 2
    ids = tokenizer(sentence, add_special_tokens=False)["input_ids"]
    step = content_capacity - stride

    pieces = []
    i = 0
    while True:
        pieces.append(tokenizer.decode(ids[i:i + content_capacity]))
        if i + content_capacity >= len(ids):
            break
        i += step

    return pieces


def chunk_by_sentences(
    text: str,
    tokenizer,
    max_length: int = 150,
    stride: int = 30,
    sentence_overlap: int = 1,
) -> list[str]:
    """문장 경계를 지키면서 각 청크가 max_length 토큰(특수 토큰 포함) 이내가 되도록 나눈다.

    한 문장이 통째로 content_capacity를 넘는 run-on sentence는 _split_long_sentence로
    토큰 단위 sliding-window fallback을 적용해, 뒷부분이 조용히 잘려나가지 않게 한다.
    """
    content_capacity = max_length - 2  # [CLS], [SEP]

    sentences = sent_tokenize(text)
    if not sentences:
        return [text]

    sent_lens = [
        len(tokenizer(s, add_special_tokens=False)["input_ids"])
        for s in sentences
    ]

    chunks = []
    n = len(sentences)
    start = 0

    while start < n:
        # 문장 하나가 이미 capacity를 넘는 경우: 그 문장만 별도로 token-level fallback 처리
        if sent_lens[start] > content_capacity:
            chunks.extend(
                _split_long_sentence(sentences[start], tokenizer, max_length, stride)
            )
            start += 1
            continue

        end = start
        total = 0
        while (
            end < n
            and sent_lens[end] <= content_capacity
            and total + sent_lens[end] <= content_capacity
        ):
            total += sent_lens[end]
            end += 1

        chunks.append(" ".join(sentences[start:end]))

        if end >= n:
            break

        # 다음 청크는 이번 청크의 마지막 sentence_overlap개 문장부터 다시 시작 (문장 단위 overlap)
        start = max(start + 1, end - sentence_overlap)

    return chunks

In [13]:
def predict_single_text_sentence_chunked(
    text: str,
    model: nn.Module,
    tokenizer,
    device: torch.device,
    max_length: int = 150,
    stride: int = 30,
    sentence_overlap: int = 1,
) -> float:
    chunks = chunk_by_sentences(
        text,
        tokenizer,
        max_length=max_length,
        stride=stride,
        sentence_overlap=sentence_overlap,
    )

    encoded = tokenizer(
        chunks,
        add_special_tokens=True,
        max_length=max_length,
        truncation=True,
        padding="max_length",
        return_attention_mask=True,
        return_token_type_ids=True,
        return_tensors="pt"
    )
    encoded = {key: value.to(device) for key, value in encoded.items()}

    with torch.no_grad():
        logits = model(**encoded)  # (num_chunks, 2)
        probabilities = F.softmax(logits, dim=1)

    return probabilities[:, 1].max().item()

In [14]:
len(df)

151978

### Sampling 예시

In [16]:
df_sample = df.sample(100, random_state=0)

In [17]:
from tqdm.auto import tqdm

tqdm.pandas(desc="predicting authority scores (sentence-level chunking + max-pooling)")
df_sample["authority_comment"] = df_sample["processed_body"].progress_apply(
    lambda text: predict_single_text_sentence_chunked(
        text,
        model=model,
        tokenizer=tokenizer,
        device=device
    )
)

predicting authority scores (sentence-level chunking + max-pooling):   0%|          | 0/100 [00:00<?, ?it/s]

In [19]:
from tqdm.auto import tqdm

# the list of Moral (MFT) values
mft_values = ["care", "harm", "fairness", "cheating", "loyalty", "betrayal",
              "authority", "subversion", "purity", "degradation"]

for mft in mft_values:
  repo_name = f"vjosap/moralBERT-predict-{mft}-in-text"
  model = MoralBERTModel.from_pretrained(repo_name, bert_model=bert_model)
  model.to(device)
  model.eval()

  tqdm.pandas(desc=f"predicting {mft} scores (sentence-level chunking + max-pooling)")
  df_sample[f"{mft}_comment"] = df_sample["processed_body"].progress_apply(
    lambda text: predict_single_text_sentence_chunked(
        text,
        model=model,
        tokenizer=tokenizer,
        device=device
    )
    )
df_sample

config.json:   0%|          | 0.00/22.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

predicting care scores (sentence-level chunking + max-pooling):   0%|          | 0/100 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/22.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

...

predicting purity scores (sentence-level chunking + max-pooling):   0%|          | 0/100 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/22.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

predicting degradation scores (sentence-level chunking + max-pooling):   0%|          | 0/100 [00:00<?, ?it/s]

,thread_id,OP,post_id,thread_size,comment_depth,is_delta,thread_pattern,ChallengerOP,comment_id,author,...,authority_comment,care_comment,harm_comment,fairness_comment,cheating_comment,loyalty_comment,betrayal_comment,subversion_comment,purity_comment,degradation_comment
63873,67aa4ff60bc679138ffb3439,faultinmystructure,bwfa5z,1.0,1,0,C,C,678542b9f5757a629ba39a5f,SnarkElemental,...,0.000727,0.113235,0.000725,0.000377,0.000638,0.000899,0.000407,0.000697,0.001335,0.011527
52445,67aa4fd30bc679138ffaadf5,LeagueSucksLol,bjzyk0,5.0,1,1,COCOD,C,678542a6f5757a629ba22110,IlluminatusUIUC,...,0.001046,0.029578,0.001257,0.012855,0.005551,0.001097,0.000864,0.000716,0.000686,0.006711
12875,67aa4f5d0bc679138ff8fa59,I_Like_Triscuits,aksn2r,5.0,4,0,COCOC,O,6785426ef5757a629b9dd4a3,I_Like_Triscuits,...,0.000809,0.400050,0.001882,0.973770,0.083387,0.001093,0.000998,0.000469,0.000790,0.007800
122964,67aa50c40bc679138ffdf638,Hey-I-Read-It,dm0m0f,1.0,1,0,C,C,67854310f5757a629baa7734,SachemNiebuhr,...,0.000841,0.073902,0.032501,0.840419,0.034171,0.002877,0.006244,0.004014,0.000604,0.015292
88920,67aa50510bc679138ffc585e,tjmaxal,ckxm9q,3.0,3,1,COD,D,678542dcf5757a629ba6658e,DeltaBot,...,0.001214,0.013238,0.000793,0.001315,0.001084,0.018622,0.008561,0.001050,0.001411,0.019161
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83341,67aa503a0bc679138ffc1746,fookhar,cg34lu,5.0,3,0,COCOC,C,678542d4f5757a629ba5c5aa,Crayshack,...,0.000396,0.064738,0.000544,0.000668,0.000468,0.001128,0.000430,0.000495,0.000670,0.006788
5517,67aa4f420bc679138ff8a0b3,teacherofderp,afjw98,7.0,6,0,COCOCOC,O,67854263f5757a629b9cf86b,teacherofderp,...,0.006165,0.043528,0.001376,0.768517,0.046292,0.005062,0.000625,0.000625,0.013214,0.013245
75455,67aa501b0bc679138ffbb6d7,mrstackz,c94o0l,2.0,1,0,CO,C,678542c8f5757a629ba4cf8f,goodguygreenpepper,...,0.000434,0.007728,0.000741,0.000297,0.000520,0.001482,0.002700,0.001157,0.000516,0.024104
119621,67aa50b70bc679138ffdcb2b,EdominoH,diaeag,4.0,3,0,COCO,C,6785430af5757a629baa0953,AnythingApplied,...,0.000995,0.026562,0.001292,0.000397,0.001306,0.001642,0.004888,0.000692,0.000736,0.089055
